In [0]:
from pyspark.sql.functions import *


1.Create a DataFrame with 5 rows containing columns: employee_id, name, department, salary, and join_date.

In [0]:
data = [
    (1, 'John','Data Analyst',40000,'03/03/2025'),
    (2, 'Mary','Data Scientist',50000,'04/04/2025'),
    (3, 'Mike','Data Engineer',60000,'02/03/2025'),
    (4,'Prabhu','Data Engineer',70000,'01/03/2025'),
    (5,'Priya','Data Scientist',80000,'05/03/2025')
]
df = spark.createDataFrame(data, ['id', 'name', 'role', 'salary', 'date'])
df.show()

2.Select only the name and salary columns from the DataFrame.

In [0]:
df.select('name','salary').show()

3.Rename the column join_date to date_of_joining.

In [0]:
df2= df.withColumnRenamed('date','date_of_joining')
df2.show()

4.Add a new column annual_bonus calculated as 10% of the salary.

In [0]:
df3 = df2.withColumn('annual_bonus', col('salary')*0.1)
df3.show()


### Set 2: Filtering & Aggregations

5.Filter the DataFrame to show employees who belong to the "Engineering" department AND have a salary greater than 75,000.

In [0]:
df3.where((col('salary') > 75000) & (col('role') == 'Data Scientist')).show()

6. Find the total number of employees and the average salary for each department.

In [0]:
df3.groupBy('role').agg((avg('salary').alias("avg_salary")),(count('id').alias("count"))).show()

7.Find the maximum salary in the entire dataset without using groupBy().

In [0]:
# Find the row with maximum salary
df3.filter(col('salary') == df3.agg(max('salary')).collect()[0][0]).show()

In [0]:
df3.agg(max('salary').alias('max salary')).show()

### Set 3: Joins & Window Functions

8.Given a second DataFrame containing department and location, perform an inner join with your employee DataFrame to add the location for each employee.

In [0]:
data2 = [
    ("Data Scientist", "Banglore"),
    ("Data Engineer","Hyderabad"),
    ("Data Analyst", "Chennai")
]
dfv2 = spark.createDataFrame(data2, ["role", "city"])
dfv2.show()

In [0]:
joined_df = df3.join(dfv2, on="role", how="inner")
joined_df.show()

9. Use a window function to rank employees by salary within each department (highest salary = Rank 1).

In [0]:
from pyspark.sql import Window

In [0]:
df3.withColumn("rank", row_number().over(Window.partitionBy(df3["role"]).orderBy(col("salary").desc()))).show()

10.Calculate the difference between each employee's salary and the average salary of their respective department.

In [0]:
df3.withColumn("avg_dep_salary", avg('salary').over(Window.partitionBy(df3["role"]))) \
   .withColumn("difference", col("salary") - col("avg_dep_salary")) \
   .show()

###### creating separate schema for practicing pyspark

In [0]:
%sql
create schema if not exists brazilian_e_commerce.pyspark_practice;

### Task 1: Ingestion, Inspection & Basic Manipulation

###### 1.Read olist_orders_dataset.csv from the volume into a DataFrame called orders_df with header=True and inferSchema=True.

In [0]:
order_df = spark.read.csv("/Volumes/brazilian_e_commerce/pyspark_practice/brazilian-ecommerce/olist_orders_dataset.csv", header=True, inferSchema=True)
order_df.show()

###### 2.Inspect the schema using .printSchema() and check the total row count.

In [0]:
order_df.printSchema()

In [0]:
order_df.count()

###### 3.Select order_id, customer_id, order_status, and order_purchase_timestamp. 

In [0]:
t1_order_df = order_df.select(col("order_id"),col("customer_id"),col("order_status"),col("order_purchase_timestamp"))
t1_order_df.show()

###### 4.Filter for orders where order_status is not 'delivered'.

In [0]:
t1_order_df.filter(col("order_status") != "delivered").show()